In [5]:
# Manipulação de dados
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Construção de pipelines e aplicação de transformações por coluna
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Pré-processamento numérico e categórico
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, PowerTransformer, OrdinalEncoder

# Codificação de variáveis categóricas nominais
import category_encoders as ce

In [7]:
df = pd.read_csv(r"..\data\processed\V1\Feature_engineering.csv")


# A variável alvo apresentou assimetria à direita, com poucos imóveis muito caros
# A transformação logarítmica reduz esse efeito e costuma melhorar modelos de regressão
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]
y_log = np.log1p(y)


SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_log,
    test_size=0.15,
    random_state=SEED
)

X_train = X_train.copy()
X_test = X_test.copy()



#-------------------- Tratamento de valores ausentes -------------------------

# Em várias variáveis categóricas, NaN representa ausência da característica,
# Preencher como "None" preserva essa informação para o modelo
cols_cat_ausencia = X_train.select_dtypes(include="object").columns.tolist()

X_train[cols_cat_ausencia] = X_train[cols_cat_ausencia].fillna("None")
X_test[cols_cat_ausencia] = X_test[cols_cat_ausencia].fillna("None")


# Para essa variável, o nulo também indica ausência.
# Portanto, zero é uma escolha mais adequada.
# A verificação evita erro caso MasVnrArea tenha sido removida no feature engineering.
if "MasVnrArea" in X_train.columns:
    X_train["MasVnrArea"] = X_train["MasVnrArea"].fillna(0)
    X_test["MasVnrArea"] = X_test["MasVnrArea"].fillna(0)


# LotFrontage foi removida depois do feature engineering.
# Por isso, o tratamento manual por mediana de Neighborhood não deve mais ser feito aqui.
# A informação dela já foi parcialmente aproveitada na criação de LotFrontageRatio.



# -------------------- Separação dos grupos de variáveis --------------------

# Essas variáveis têm ordem natural de qualidade ou condição;
# A ordem precisa ser definida manualmente, porque a ordem alfabética não representa
# corretamente a escala de qualidade do dataset
cols_ord = [
    'ExterQual', 'ExterCond',
    'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual',
    'FireplaceQu',
    'GarageQual', 'GarageCond'
]

# Mantém apenas as colunas ordinais que ainda existem após o feature engineering
cols_ord = [
    col for col in cols_ord
    if col in X_train.columns
]

ordem_qualidade = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

'''
None = ausência da característica
Po   = Poor / ruim
Fa   = Fair / razoável
TA   = Typical / mediano
Gd   = Good / bom
Ex   = Excellent / excelente
'''

# Lista com listas da ordem de qualidade para cada variável ordinal do dataset
categorias_ordinais = [ordem_qualidade for _ in cols_ord]


# As demais categóricas serão tratadas como nominais
# Como há muitas variáveis categóricas, TargetEncoder evita criar muitas colunas,
# diferente do OneHotEncoder.
cols_cat = [
    col for col in X_train.select_dtypes(include=['object']).columns
    if col not in cols_ord
]


# Variáveis binárias criadas no feature engineering.
# Elas representam presença/ausência ou flags lógicas, então não faz sentido aplicar PowerTransformer nelas.
cols_binary = [
    col for col in X_train.select_dtypes(include=['int', 'float']).columns
    if X_train[col].dropna().isin([0, 1]).all()
    and X_train[col].nunique(dropna=True) <= 2
]


# Variáveis numéricas assimétricas.
# Calculamos a assimetria apenas nas numéricas que não são binárias.
cols_num_candidates = [
    col for col in X_train.select_dtypes(include=['int', 'float']).columns
    if col not in cols_binary
]

skewness = X_train[cols_num_candidates].skew(numeric_only=True)

cols_skew = skewness[
    skewness.abs() > 1
].index.tolist()


# Demais variáveis numéricas serão tratadas com RobustScaler,
# pois o dataset possui variáveis com outliers.
cols_num = [
    col for col in cols_num_candidates
    if col not in cols_skew
]



# -------------------- Pipelines de pré-processamento --------------------

transformer_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])


transformer_ordinal = Pipeline(steps=[
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='None'
    )),
    ('encoder', OrdinalEncoder(
        categories=categorias_ordinais,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])


# Yeo-Johnson foi escolhido porque lida com assimetria e aceita valores zero,
# já que nesse dataset várias variáveis numéricas têm muitos zeros
transformer_skew = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log', PowerTransformer(method='yeo-johnson'))
])


# Variáveis binárias não precisam de escala nem transformação.
# Apenas imputamos o valor mais frequente caso exista algum nulo.
transformer_binary = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])


transformer_cat = Pipeline(steps=[
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='None'
    )),
    ('target', ce.TargetEncoder(
        smoothing=10,
        handle_unknown='value',
        handle_missing='value'
    ))
])


# -------------------- Pré-processador final --------------------

preprocessor = ColumnTransformer([
    ('num', transformer_numerico, cols_num),
    ('ord', transformer_ordinal, cols_ord),
    ('skew', transformer_skew, cols_skew),
    ('binary', transformer_binary, cols_binary),
    ('cat', transformer_cat, cols_cat)
])

C:\Users\Gabryel\AppData\Local\Temp\ipykernel_12868\1357601812.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_cat_ausencia = X_train.select_dtypes(include="object").columns.tolist()
C:\Users\Gabryel\AppData\Local\Temp\ipykernel_12868\1357601812.py:87: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas

In [8]:
# Primeiro, o pré-processador aprende os padrões apenas do conjunto de treino
# e já aplica essas transformações no próprio treino
X_train_preprocessado = preprocessor.fit_transform(X_train, y_train)

# Depois, aplicamos no teste exatamente as mesmas regras aprendidas no treino
# Aqui usamos apenas transform, porque o teste deve simular dados novos,
# sem ensinar nada ao pré-processador 
# Assim evitamos data leakage
X_test_preprocessado = preprocessor.transform(X_test)

nomes_colunas = (
    [f'num__{col}' for col in cols_num] +
    [f'ord__{col}' for col in cols_ord] +
    [f'skew__{col}' for col in cols_skew] +
    [f'cat__{col}' for col in cols_cat]
)

X_train_preprocessado = pd.DataFrame(
    X_train_preprocessado,
    columns=nomes_colunas,
    index=X_train.index
)

X_test_preprocessado = pd.DataFrame(
    X_test_preprocessado,
    columns=nomes_colunas,
    index=X_test.index
)

In [9]:
X_train_preprocessado.to_csv(r"..\data\processed\V1\X_processed_train.csv", index=False)
X_test_preprocessado.to_csv(r"..\data\processed\V1\X_processed_test.csv", index=False)

# -------------------- Salvando variável alvo processada --------------------

# y_train e y_test já estão na escala logarítmica, pois o split foi feito usando y_log.
# Salvamos separadamente para manter claro quais dados pertencem ao treino e ao teste.
y_train.to_csv(
    r"..\data\processed\V1\y_processed_train.csv",
    index=False,
    header=["SalePrice_log"]
)

y_test.to_csv(
    r"..\data\processed\V1\y_processed_test.csv",
    index=False,
    header=["SalePrice_log"]
)